Step 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Load the dataset (Use a sample if memory is an issue)
# df = pd.read_csv('Data/US_Accidents_March23.csv')
# For Google Colab, you might need to upload to your Drive first:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/US_Accidents_March23.csv')

# --- DATA PREPARATION ---
def prepare_data(df):
    # 1. Convert to Datetime
    df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')

    # 2. Extract Temporal Features
    df['Hour'] = df['Start_Time'].dt.hour
    df['Weekday'] = df['Start_Time'].dt.day_name()
    df['Month'] = df['Start_Time'].dt.month

    # 3. Handle Missing Values
    # Drop columns with > 30% missing data
    limit = len(df) * 0.7
    df = df.dropna(thresh=limit, axis=1)

    # Fill remaining numerical gaps with median
    df = df.fillna(df.median(numeric_only=True))

    return df

# df = prepare_data(df)
print("Data Preparation Complete.")

Step 3

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

# 1. Hourly Pattern (When)
sns.histplot(df['Hour'], bins=24, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Accident Frequency by Hour of Day')
axes[0].set_xticks(range(24))

# 2. Weekly Pattern (Pattern)
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(data=df, x='Weekday', order=order, ax=axes[1], palette='magma')
axes[1].set_title('Accident Frequency by Day of Week')

plt.tight_layout()
plt.show()

Step 4

In [ ]:
# Question: Is there a significant relationship between Sunrise/Sunset and Severity?
contingency = pd.crosstab(df['Sunrise_Sunset'], df['Severity'])
chi2, p, dof, ex = chi2_contingency(contingency)

print(f"Chi-Square Result: {chi2:.2f}")
print(f"P-value: {p:.4f}")

if p < 0.05:
    print("Insight: Time of day (Light vs Dark) significantly impacts accident severity.")

Step 5

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure dates are in datetime format (from Step 4)
df['Start_Time'] = pd.to_datetime(df['Start_Time'])
df['Hour'] = df['Start_Time'].dt.hour
df['Weekday'] = df['Start_Time'].dt.day_name()

# Set the visual style
sns.set_theme(style="whitegrid")

# --- Visualization 1: Hourly Distribution (The "When") ---
plt.figure(figsize=(12, 6))
sns.histplot(df['Hour'], bins=24, kde=True, color='skyblue')
plt.title('Distribution of Accidents by Hour of Day', fontsize=15)
plt.xlabel('Hour (0-23)', fontsize=12)
plt.ylabel('Frequency of Accidents', fontsize=12)
plt.xticks(range(0, 24))
plt.show()
# Interpretation: Look for peaks during morning (7-9 AM) and evening (4-6 PM) rush hours.

# --- Visualization 2: Severity vs. Points of Interest (The "Where") ---
# Analyzing if infrastructure features like Crossings or Signals affect Severity
poi_cols = ['Crossing', 'Junction', 'Traffic_Signal', 'Railway']
poi_counts = df[poi_cols].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=poi_counts.index, y=poi_counts.values, palette='viridis')
plt.title('Frequency of Accidents Near Infrastructure Features', fontsize=15)
plt.ylabel('Number of Accidents', fontsize=12)
plt.show()
# Interpretation: Identifies which infrastructure types require the most safety audits.

# --- Visualization 3: Accidents by Day of Week (The "Pattern") ---
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
plt.figure(figsize=(12, 6))
sns.countplot(data=df, x='Weekday', order=weekday_order, palette='magma')
plt.title('Accident Frequency by Day of the Week', fontsize=15)
plt.xlabel('Day', fontsize=12)
plt.ylabel('Number of Accidents', fontsize=12)
plt.show()
# Interpretation: Usually shows a significant drop on weekends, highlighting work-commute risks.

Step 6

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. Correlation Analysis (Numerical Relationships) ---
# We check if features like Humidity, Pressure, or Temperature correlate with Severity
# Note: Severity is often treated as numerical for correlation, though it's technically ordinal
numerical_cols = ['Severity', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)']
corr_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Environmental Factors vs. Severity')
plt.show()

# --- 2. Chi-Square Test: Weather Condition vs. Severity ---
# Question: Does the Weather_Condition have a significant relationship with Accident Severity?

# Create a contingency table
contingency_table = pd.crosstab(df['Weather_Condition'], df['Severity'])

# Perform Chi-Square Test
chi2, p, dof, expected = chi2_contingency(contingency_table)

print("--- Chi-Square Test Results (Weather vs. Severity) ---")
print(f"Chi-Square Statistic: {chi2:.2f}")
print(f"P-value: {p:.4f}")

if p < 0.05:
    print("Conclusion: There is a statistically significant relationship between Weather and Severity.")
else:
    print("Conclusion: No significant relationship found.")

# --- 3. T-Test: Night vs. Day Severity ---
# Question: Are accidents significantly more severe at night?
from scipy.stats import ttest_ind

day_accidents = df[df['Sunrise_Sunset'] == 'Day']['Severity']
night_accidents = df[df['Sunrise_Sunset'] == 'Night']['Severity']

t_stat, p_val_t = ttest_ind(day_accidents.dropna(), night_accidents.dropna())

print("\n--- T-Test Results (Day vs. Night Severity) ---")
print(f"P-value: {p_val_t:.4f}")
if p_val_t < 0.05:
    print("Conclusion: The difference in severity between Day and Night is statistically significant.")

Step 7

In [ ]:
# 1. Recommendation 1: Temporal Resource Allocation
# Justification: Find the exact peak hour for high-severity accidents
high_severity = df[df['Severity'] >= 3]
peak_hour = high_severity['Hour'].mode()[0]
peak_count = high_severity['Hour'].value_counts().max()

print(f"--- Recommendation 1: Peak Hour Enforcement ---")
print(f"Data Insight: Severity 3 & 4 accidents peak at {peak_hour}:00.")
print(f"Action: Deploy emergency response units to highway junctions at {peak_hour-1}:45 to reduce clearance time.")

# 2. Recommendation 2: Infrastructure Safety Audits
# Justification: Which POI (Point of Interest) is most dangerous?
poi_cols = ['Crossing', 'Junction', 'Traffic_Signal', 'Railway', 'Station', 'Stop']
poi_severity = df.groupby('Severity')[poi_cols].sum().T
top_poi = poi_severity.sum(axis=1).idxmax()

print(f"\n--- Recommendation 2: Infrastructure Audit ---")
print(f"Data Insight: '{top_poi}' features are present in the highest number of accidents.")
print(f"Action: Audit signal timing and visibility at all '{top_poi}' locations in high-density counties.")

# 3. Recommendation 3: Weather-Based Speed Regulation
# Justification: Compare Mean Severity in bad weather vs clear weather
weather_impact = df.groupby('Weather_Condition')['Severity'].mean().sort_values(ascending=False)

print(f"\n--- Recommendation 3: Dynamic Speed Limits ---")
print(f"Data Insight: Top 3 deadliest weather conditions: {', '.join(weather_impact.index[:3])}")
print(f"Action: Implement IoT-enabled dynamic speed limit signs that lower limits during these specific weather events.")